# L69 · CTC 前向算法 — 所有路径概率求和，O(T×2L) 纯 NumPy 实现

**学习目标**
- 理解 CTC 的核心挑战：不知道字符在哪一帧对齐
- 掌握前向变量 α 的递推定义
- 用纯 NumPy 实现 CTC 前向算法并数值验证

## CTC 的核心问题

声学模型输出 T 帧，标签序列有 L 个字符。帧数 T >> L，但不知道每个字符对应哪几帧。

CTC 解法：引入空白符 `<blank>`，允许所有合法的对齐路径，然后对它们的概率求和。

**合法路径** = 任意长度为 T 的序列，折叠后（去掉空白、合并重复）等于目标标签。

**扩展标签** l' = `<b> l₁ <b> l₂ <b> ... <b> lL <b>`，长度 S = 2L+1。

**前向变量** α[t][s] = 在时刻 t，所有到达扩展标签位置 s 的路径前缀概率之和（log 域）。

**递推（三个合法前驱）**：
```
α[t][s] = ( α[t-1][s]              # 停留在 s
          + α[t-1][s-1]            # 从 s-1 跳来
          + α[t-1][s-2]  {仅当 l'[s] ≠ l'[s-2]}  # 跳过中间 blank
          ) × p[t][l'[s]]
```

In [ ]:
import numpy as np

In [ ]:
# 构造玩具示例：3 帧，词汇 {a:0, b:1, blank:2}，目标 'ab'
BLANK = 2
VOCAB_SIZE = 3
T = 6
LABELS = [0, 1]   # 'a', 'b'

np.random.seed(42)
logits = np.random.randn(T, VOCAB_SIZE)
# 转为 log 概率
log_probs = logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))
print(f'log_probs shape: {log_probs.shape}  (T={T}, vocab={VOCAB_SIZE})')

## ✏️ 实现 CTC 前向算法

In [ ]:
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """CTC 前向算法：计算 log P(labels | log_probs)。

    Args:
        log_probs: (T, vocab_size) log-probability matrix.
        labels:    list of integer label ids (without blanks).
        blank:     blank token index.

    Returns:
        log probability of the label sequence under CTC.
    """
    T, V = log_probs.shape
    L = len(labels)

    # 扩展标签：blank + label + blank + label + ... + blank
    lprime = [blank]
    for c in labels:
        lprime.append(c)
        lprime.append(blank)
    S = len(lprime)   # = 2L + 1

    NEG_INF = -1e30
    alpha = np.full((T, S), NEG_INF)

    # ✏️ TODO: 初始条件（第 0 帧只能从位置 0 或位置 1 出发）
    alpha[0][0] = log_probs[0][lprime[0]]
    if S > 1:
        alpha[0][1] = log_probs[0][lprime[1]]

    def log_add(a, b):
        if a == NEG_INF: return b
        if b == NEG_INF: return a
        m = max(a, b)
        return m + np.log(np.exp(a - m) + np.exp(b - m))

    # ✏️ TODO: 递推填充 alpha[t][s]，t=1..T-1
    for t in range(1, T):
        for s in range(S):
            val = alpha[t-1][s]                            # 前驱 1: 停留
            if s >= 1:
                val = log_add(val, alpha[t-1][s-1])        # 前驱 2: 跳 1
            if s >= 2 and lprime[s] != lprime[s-2]:
                val = log_add(val, alpha[t-1][s-2])        # 前驱 3: 跳 2
            alpha[t][s] = val + log_probs[t][lprime[s]]

    # 结束状态：最后的 blank（S-1）或最后的标签（S-2）
    return float(log_add(alpha[T-1][S-1], alpha[T-1][S-2]))

In [ ]:
lp = ctc_forward(log_probs, LABELS, blank=BLANK)
print(f'log P(labels | frames) = {lp:.4f}')
print(f'P(labels | frames)     = {np.exp(lp):.8f}')
assert np.isfinite(lp), '前向算法应返回有限值'
assert lp < 0, 'log 概率应为负数'
print('✅ CTC 前向算法验证通过')

## 复杂度与直觉

In [ ]:
# 展示：T 增大时计算量线性增长（而不是指数增长）
print(f'{"T":>4}  {"S=2L+1":>6}  {"计算格数":>8}  {"log P":>8}')
print('-' * 35)
for T_test in [5, 10, 20, 50, 100]:
    lp_test = np.random.randn(T_test, VOCAB_SIZE)
    lp_test -= np.log(np.exp(lp_test).sum(axis=1, keepdims=True))
    result = ctc_forward(lp_test, LABELS, blank=BLANK)
    S = 2*len(LABELS)+1
    print(f'{T_test:>4}  {S:>6}  {T_test*S:>8}  {result:>8.3f}')

## 小结

| 概念 | 要记住的 |
|---|---|
| 扩展标签 l' | blank 插在每个标签之间，长度 2L+1 |
| 前向变量 α[t][s] | log 域递推，避免下溢 |
| 三个前驱 | 停留、跳 1、跳 2（限字符不同时）|
| 复杂度 | O(T × 2L) — 与暴力枚举路径的指数复杂度相比快得多 |
| L70 | Whisper 用 Attention 解码，但仍然是 token-by-token 的自回归过程 |